# Сравнение XGBoost, CatBoost и LightGBM

Цель — сравнить три градиентных бустинга для прогноза `Rented Bike Count`. Подготовка данных, признаки и хронологическое разбиение приведены к `hyperparameter-tuning_team13.ipynb`: версия датасета 1, 70% train / 15% validation / последние 15% test, `TimeSeriesSplit(n_splits=4, gap=24)` внутри train.

Используются 17 признаков из ноутбука тюнинга: исходные час и погодные показатели, категориальные признаки, `day_of_week`, `month`, `is_weekend`, `hour_sin` и `hour_cos`. Как и в ноутбуке тюнинга, `day_of_week` и `month` трактуются как числовые; EDA предлагает отдельно проверить категориальный `day_of_week` и сравнить исходный час с циклическим кодированием. `day_type` и расстояния до пиков тоже остаются отдельными экспериментами.

Числовые признаки не масштабируются. Test не участвует ни в подборе, ни в сравнении.

## 1. Импорты и параметры

In [ ]:
import os
from copy import deepcopy

import kagglehub
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

RANDOM_STATE = 42
N_SPLITS = 4
CV_GAP = 24

## 2. Загрузка и подготовка данных

Используется фиксированная версия датасета из ноутбука тюнинга. Временная отметка нужна для сортировки и проверки, но не подаётся в модели.

In [ ]:
CSV_PATH = os.path.join(
    kagglehub.dataset_download("joebeachcapital/seoul-bike-sharing/versions/1"),
    "SeoulBikeData.csv",
)
try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, encoding="cp1252")

required_columns = {
    "Date", "Hour", "Rented Bike Count", "Seasons", "Holiday", "Functioning Day"
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"В CSV отсутствуют колонки: {missing_columns}")
if len(df) != 8760:
    raise ValueError(f"Ожидалось 8760 строк, получено {len(df)}")

df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df["timestamp"] = df["Date"] + pd.to_timedelta(df["Hour"], unit="h")
df = df.sort_values("timestamp").reset_index(drop=True)
if df["timestamp"].duplicated().any():
    raise ValueError("В данных есть повторяющиеся дата + час")

df["day_of_week"] = df["Date"].dt.day_of_week
df["month"] = df["Date"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype("int8")
df["hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

print(f"Размер датасета: {df.shape}")
print(f"Период: {df['timestamp'].min()} — {df['timestamp'].max()}")

## 3. Хронологическое разбиение

Разбиение совпадает с `hyperparameter-tuning_team13.ipynb`: первые 70% — train, следующие 15% — validation, последние 15% — отложенный test. Нерабочие часы сохраняются.

In [ ]:
test_size = int(len(df) * 0.15)
validation_size = int(len(df) * 0.15)
train_size = len(df) - validation_size - test_size

train_df = df.iloc[:train_size].copy()
validation_df = df.iloc[train_size : train_size + validation_size].copy()
test_df = df.iloc[train_size + validation_size :].copy()

target = "Rented Bike Count"
categorical_features = ["Seasons", "Holiday", "Functioning Day"]
numeric_features = [
    "Hour", "Temperature(°C)", "Humidity(%)", "Wind speed (m/s)",
    "Visibility (10m)", "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)", "Rainfall(mm)", "Snowfall (cm)",
    "day_of_week", "month", "is_weekend", "hour_sin", "hour_cos",
]
features = numeric_features + categorical_features
missing_features = set(features + [target]) - set(df.columns)
if missing_features:
    raise ValueError(f"Не найдены признаки: {missing_features}")

X_train = train_df[features]
y_train = train_df[target]
X_validation = validation_df[features]
y_validation = validation_df[target]

print(
    f"Train: {len(train_df)}, validation: {len(validation_df)}, "
    f"test (отложен): {len(test_df)}"
)
print(f"Признаков: {len(features)}; X_train: {X_train.shape}; "
      f"X_validation: {X_validation.shape}")

## 4. Препроцессинг и модели

XGBoost и LightGBM получают One-Hot-кодирование категорий; известные уровни заданы явно, чтобы категория сезона, отсутствующая в train, не теряла свой признак на validation. CatBoost получает исходные категории и обрабатывает их нативно. Для всех моделей используются одни и те же строки и целевая переменная.

In [ ]:
category_levels = [
    ["Autumn", "Spring", "Summer", "Winter"],
    ["Holiday", "No Holiday"],
    ["No", "Yes"],
]
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                categories=category_levels,
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
        ("numeric", "passthrough", numeric_features),
    ]
)

models = {
    "XGBoost": make_pipeline(
        preprocessor,
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        ),
    ),
    "LightGBM": make_pipeline(
        preprocessor,
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    "CatBoost": CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        cat_features=categorical_features,
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    ),
}

print("Модели подготовлены:", ", ".join(models))

## 5. Кросс-валидация и сравнение на validation

Используем тот же `TimeSeriesSplit(n_splits=4, gap=24)`, что и в ноутбуке тюнинга. Каждый fold обучается только на прошлом; после выбора настроек алгоритма модель один раз оценивается на validation. Test не трогаем.

In [ ]:
time_series_cv = TimeSeriesSplit(n_splits=N_SPLITS, gap=CV_GAP)
results = []
fitted_models = {}

for model_name, estimator in models.items():
    fold_rmse = []

    for train_indices, fold_indices in time_series_cv.split(X_train):
        fold_model = deepcopy(estimator)
        X_fold_train = X_train.iloc[train_indices]
        y_fold_train = y_train.iloc[train_indices]
        X_fold_validation = X_train.iloc[fold_indices]
        y_fold_validation = y_train.iloc[fold_indices]

        fold_model.fit(X_fold_train, y_fold_train)
        fold_predictions = fold_model.predict(X_fold_validation)
        fold_rmse.append(
            np.sqrt(mean_squared_error(y_fold_validation, fold_predictions))
        )

    final_model = deepcopy(estimator)
    final_model.fit(X_train, y_train)
    validation_predictions = final_model.predict(X_validation)
    fitted_models[model_name] = final_model

    results.append(
        {
            "model": model_name,
            "cv_rmse_mean": np.mean(fold_rmse),
            "cv_rmse_std": np.std(fold_rmse),
            "validation_rmse": np.sqrt(
                mean_squared_error(y_validation, validation_predictions)
            ),
            "validation_mae": mean_absolute_error(
                y_validation, validation_predictions
            ),
            "validation_r2": r2_score(y_validation, validation_predictions),
        }
    )

    print(f"{model_name}: CV RMSE = {np.mean(fold_rmse):.2f} "
          f"+/- {np.std(fold_rmse):.2f}")

results_df = pd.DataFrame(results).sort_values("validation_rmse").reset_index(drop=True)
results_df.round(3)

## 6. Результаты

| Модель | CV RMSE, mean ± std | Validation RMSE | Validation MAE | Validation R² |
| --- | ---: | ---: | ---: | ---: |
| CatBoost | 352.96 ± 131.01 | **284.42** | **198.76** | **0.8418** |
| XGBoost | **349.89 ± 130.45** | 309.63 | 205.88 | 0.8125 |
| LightGBM | 355.11 ± 140.96 | 339.41 | 213.13 | 0.7746 |

На validation лидирует CatBoost по RMSE/MAE/R²; у XGBoost немного ниже средний CV RMSE. Разница средних CV RMSE невелика относительно разброса между временными фолдами, поэтому результаты остаются предварительными и требуют тюнинга выбранной модели. Test не использовался.

## 7. Итог

Ниже код повторно выводит лидера по validation RMSE. Это сравнение моделей с начальными параметрами; следующий шаг — тюнинг выбранной модели. Test следует открыть только для одной итоговой оценки после выбора.

In [ ]:
best_result = results_df.iloc[0]
print(
    f"Лучшая по validation RMSE модель: {best_result['model']} "
    f"(RMSE={best_result['validation_rmse']:.2f}, "
    f"R2={best_result['validation_r2']:.4f})."
)
print("Test-метрики не рассчитаны и не использовались для выбора.")